## Lab: Semantic Search and Mini-RAG

Use the 10 test queries below to compare retrieval-grounded answers against the baseline on the same question. The corpus is already provided, so learners should not spend lab time collecting documents.

For each query, inspect:

- the top retrieved chunk(s),
- the baseline answer,
- the RAG answer,
- whether the RAG answer is more grounded, relevant, and complete.

If the API key is available, run both answer paths. If not, use the retrieval output to explain what context would be injected into the prompt.

In [ ]:
import sys
sys.path.append('Track1/Functions')

import json
import os
from pathlib import Path

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

from C1_4_Func import retrieve, rag_pipeline, rag_answer, non_rag_answer

corpus_path = Path("Track1") / "demos" / "data" / "C1.4_corpus" / "docs.jsonl"
if not corpus_path.exists():
    raise FileNotFoundError(f"Missing corpus file: {corpus_path}")

corpus_records = [
    json.loads(line)
    for line in corpus_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
documents = [r["text"] for r in corpus_records]
titles = [r["title"] for r in corpus_records]
document_ids = [r["id"] for r in corpus_records]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = np.asarray(embed_model.encode(documents, convert_to_numpy=True), dtype="float32")
faiss.normalize_L2(doc_embeddings)
dimension = doc_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(doc_embeddings)

try:
    from langchain_groq.chat_models import ChatGroq
    GROQ_AVAILABLE = True
except Exception as exc:
    ChatGroq = None
    GROQ_AVAILABLE = False
    groq_error = exc

groq_token = os.getenv("CHAT_GROQ_API_KEY")
llm = None
if GROQ_AVAILABLE and groq_token:
    llm = ChatGroq(groq_api_key=groq_token, model_name="llama-3.1-8b-instant")

In [ ]:
# 10 lab queries spanning AI, finance, education, and healthcare
test_queries = [
    # AI / embeddings / RAG
    "What are embeddings and why do similar texts end up close together?",
    "How does cosine similarity help compare text vectors?",
    "What is the difference between sparse search and dense search?",
    "Why does RAG reduce hallucination?",
    "What is FAISS used for?",
    # Finance
    "How does compound interest help savings grow over time?",
    "What affects my credit score and how do I improve it?",
    "Why does inflation reduce what my money can buy?",
    # Education
    "What study technique helps me remember information longer?",
    # Healthcare
    "Why is seeing a doctor before you feel sick useful?",
]

print("Retrieval preview — top-2 chunks for each lab query:")
print("=" * 72)
for i, query in enumerate(test_queries, 1):
    hits = retrieve(query, embed_model, faiss_index, document_ids, titles, documents, k=2)
    print(f"\nQ{i:02d}: {query}")
    for h in hits:
        print(f"  [{h['score']:.3f}] {h['title']}")

Interpretation: The 10 queries cover the main course ideas from embeddings through RAG. Each printed retrieval list shows which passages would be injected into the prompt before generation, which makes it easy to compare grounding quality later.

### Lab: Baseline vs RAG comparison

Run the cell below to compare answers for the first few queries. If `CHAT_GROQ_API_KEY` is set, both the baseline and RAG answers are live. Without the key the retrieval output is still shown so you can reason about what context *would* be injected into the prompt and whether that would improve the answer.

In [ ]:
# Compare baseline vs RAG for a representative cross-domain sample
# Increase sample_size or replace with your own queries to explore further
sample_queries = [
    test_queries[0],   # AI: embeddings
    test_queries[3],   # AI: RAG + hallucination
    test_queries[5],   # Finance: compound interest
    test_queries[8],   # Education: memory technique
    test_queries[9],   # Healthcare: preventive care
]

separator = "=" * 70
for query in sample_queries:
    result = rag_pipeline(query, embed_model, faiss_index, document_ids, titles, documents, k=3)
    print(separator)
    print(f"QUERY: {query}\n")
    print("Retrieved chunks:")
    for r in result["results"]:
        print(f"  [{r['score']:.3f}] {r['title']}")
    print()
    print("--- Baseline (no retrieval) ---")
    print(non_rag_answer(query, llm))
    print()
    print("--- RAG answer (with retrieved context) ---")
    print(rag_answer(query, llm, embed_model, faiss_index, document_ids, titles, documents, k=3))
    print()

print(separator)
print("Done. Check whether the RAG answers stay closer to the corpus facts.")